# PhishNet-Transformer - Step 3: Fine-Tune DistilBERT on Raw URL Text (TensorFlow/Keras)

This is the deep learning half - fine-tuning DistilBERT directly on raw URL text instead of hand-picked features. No feature engineering this time, the model figures out what matters on its own from the characters/subwords.

Using TensorFlow/Keras here (via HuggingFace's TF model classes) since that's what I already know - trained the same way as any other Keras model, model.compile() + model.fit().

Ran this in Google Colab with the free T4 GPU. Needed train.csv/val.csv/test.csv from Step 1 and xgboost_results.json from Step 2 uploaded first.

(Note: hit a couple of environment issues getting this running - a transformers v5 compatibility problem since they dropped TF support, and a safetensors loading bug. Both documented and fixed in Cell 1 and Cell 5 below.)

### Setup

In [1]:
import os
os.environ["USE_TORCH"] = "0"
os.environ["USE_TF"] = "1"

# IMPORTANT: transformers v5.0+ (released Jan 2026) removed ALL TensorFlow support.
# We pin to 4.57.6 (the last version that still has TF*/Keras classes) so this notebook
# keeps working regardless of what version Colab would install by default.
!pip install -q "transformers==4.57.6" tf-keras datasets scikit-learn

import pandas as pd
import numpy as np
import tensorflow as tf
import json
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices('GPU')) > 0)
print(tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.19.0
GPU available: True
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


GPU showed up fine, moved on.

### Load Step 1's splits again

Same exact split as the XGBoost baseline so the comparison is fair.

In [ ]:
train = pd.read_csv("train.csv")
val = pd.read_csv("val.csv")
test = pd.read_csv("test.csv")

print("Train:", train.shape, "Val:", val.shape, "Test:", test.shape)
train.head()

### Tokenize

Converting raw URLs into token IDs DistilBERT can actually process. Using max_length=64 since URLs are way shorter than normal sentences, that's plenty of room.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding='max_length',
        truncation=True,
        max_length=64,
        return_tensors="tf"
    )

train_encodings = tokenize(train['URL'])
val_encodings = tokenize(val['URL'])
test_encodings = tokenize(test['URL'])

print("input_ids shape:", train_encodings['input_ids'].shape)
print("Example input_ids (first URL):", train_encodings['input_ids'][0])

Shape came out to (8400, 64) as expected - 8400 URLs each turned into 64 tokens.

### Build tf.data.Dataset

Standard way to feed data into a Keras model for something this size, handles batching/shuffling for me.

In [ ]:
BATCH_SIZE = 32

def make_dataset(encodings, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((
        {'input_ids': encodings['input_ids'], 'attention_mask': encodings['attention_mask']},
        labels
    ))
    if shuffle:
        ds = ds.shuffle(len(labels), seed=42)
    return ds.batch(BATCH_SIZE)

train_ds = make_dataset(train_encodings, train['label'].values, shuffle=True)
val_ds = make_dataset(val_encodings, val['label'].values)
test_ds = make_dataset(test_encodings, test['label'].values)

print(train_ds)

### Load the pretrained model

num_labels=2 adds a fresh classification head (phishing/legit) on top of DistilBERT's existing language understanding - only that last layer starts untrained.

Also had to add use_safetensors=False here - ran into a known bug in the transformers library (TypeError: 'builtins.safe_open' object is not iterable) when loading safetensors weights into a TF model. This flag makes it use the older weight format instead, which loads fine.

In [ ]:
model = TFAutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, use_safetensors=False)
model.summary()

### Compile

Adam with a small learning rate (5e-5) since that's the standard for fine-tuning transformers - too high and it'll wreck the pretrained weights instead of gently adapting them.

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])
print("Model compiled.")

### Fine-tune

4 epochs, watching validation loss to catch overfitting. Added early stopping as a safety net so it automatically keeps the best checkpoint if val_loss starts going the wrong direction.

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=1,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=4,
    callbacks=[early_stopping]
)

Trained for a few epochs, val_loss kept improving so early stopping didn't need to kick in early.

### Evaluate on test set

Same test set as XGBoost, never touched during training.

In [9]:
raw_preds = model.predict(test_ds)
preds = np.argmax(raw_preds.logits, axis=1)
true_labels = test['label'].values

acc = accuracy_score(true_labels, preds)
prec = precision_score(true_labels, preds)
rec = recall_score(true_labels, preds)
f1 = f1_score(true_labels, preds)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 score:  {f1:.4f}")

cm = confusion_matrix(true_labels, preds)
cm_df = pd.DataFrame(
    cm,
    index=['Actual: legitimate', 'Actual: phishing'],
    columns=['Predicted: legitimate', 'Predicted: phishing']
)
cm_df

Accuracy:  0.9972
Precision: 1.0000
Recall:    0.9944
F1 score:  0.9972


                     Predicted: legitimate  Predicted: phishing
Actual: legitimate                     900                    0
Actual: phishing                          5                  895

### Compare against XGBoost

The actual point of this whole project - putting both models side by side.

In [10]:
with open("xgboost_results.json") as f:
    xgb_results = json.load(f)

distilbert_results = {
    "model": "DistilBERT (fine-tuned, raw URL text, TensorFlow/Keras)",
    "accuracy": float(acc),
    "precision": float(prec),
    "recall": float(rec),
    "f1": float(f1)
}

comparison = pd.DataFrame([xgb_results, distilbert_results]).set_index("model")
comparison

                                                         accuracy  precision   recall       f1
model
XGBoost (classic ML, lexical features)                  0.992778   0.996641 0.988889 0.992750
DistilBERT (fine-tuned, raw URL text, TensorFlow/Keras) 0.997222   1.000000 0.994444 0.997214

DistilBERT beat XGBoost on every metric. 99.72% accuracy vs 99.28%, and precision hit a perfect 100% - no false alarms on legit URLs at all in this test set. Out of 1800 test URLs, XGBoost got 13 wrong and DistilBERT got 5 wrong, which works out to about 61% fewer errors.

My guess for why: the transformer is probably picking up on character-level patterns (like typosquatting, weird subdomain structures) that my 12 hand-picked features didn't fully capture. has_https and num_subdirs dominated the XGBoost model, so anything outside of that pattern probably slipped through - DistilBERT reading the raw text directly doesn't have that blind spot.

### Save everything

In [11]:
# Save the fine-tuned model and tokenizer (Keras/TF format)
model.save_pretrained("./phishnet_distilbert_final")
tokenizer.save_pretrained("./phishnet_distilbert_final")

# Save results in the same format as Step 2, for consistency
with open("distilbert_results.json", "w") as f:
    json.dump(distilbert_results, f, indent=2)

# Save the combined comparison table too, so it's readable without re-running anything
comparison.to_csv("model_comparison.csv")

print("Saved:")
print("  ./phishnet_distilbert_final/  (the fine-tuned model + tokenizer)")
print("  distilbert_results.json")
print("  model_comparison.csv")
print("\nFinal comparison:")
print(comparison)

Saved:
  ./phishnet_distilbert_final/  (the fine-tuned model + tokenizer)
  distilbert_results.json
  model_comparison.csv

Final comparison:
                                                         accuracy  precision   recall       f1
model
XGBoost (classic ML, lexical features)                  0.992778   0.996641 0.988889 0.992750
DistilBERT (fine-tuned, raw URL text, TensorFlow/Keras) 0.997222   1.000000 0.994444 0.997214


### Done

DistilBERT: 99.72% accuracy, 100% precision, 99.72% F1 - beats the XGBoost baseline on every metric, about 61% fewer errors on the test set. Model + results saved, ready for the Streamlit demo next.